In [8]:
import cv2
import numpy as np

def load_images(path1, path2, size=(300, 300)):
    img1 = cv2.imread(path1)
    img2 = cv2.imread(path2)
    img1 = cv2.resize(img1, size)
    img2 = cv2.resize(img2, size)
    A = img1.astype(np.float64)
    B = img2.astype(np.float64)
    return A, B

In [9]:
def morph_color_k(A, B, steps=60, k=150):
    frames = []
    
    svd_A = [np.linalg.svd(A[:,:,c].astype(np.float64), full_matrices=False) for c in range(3)]
    svd_B = [np.linalg.svd(B[:,:,c].astype(np.float64), full_matrices=False) for c in range(3)]

    for c in range(3):
        UA, SA, VAt = svd_A[c]
        UB, SB, VBt = svd_B[c]
        for i in range(UA.shape[1]):
            if np.dot(UA[:, i], UB[:, i]) < 0:
                UB[:, i] = -UB[:, i]
                VBt[i, :] = -VBt[i, :]
        svd_B[c] = (UB, SB, VBt)
 
    for t in np.linspace(0, 1, steps):
        channels = []
        for c in range(3):
            UA, SA, VAt = svd_A[c]
            UB, SB, VBt = svd_B[c]
            Ut  = (1-t)*UA[:, :k] + t*UB[:, :k]
            St  = (1-t)*SA[:k]    + t*SB[:k]
            Vtt = (1-t)*VAt[:k,:] + t*VBt[:k,:]
            frame_c = Ut @ np.diag(St) @ Vtt
            frame_c = np.clip(frame_c, 0, 255).astype(np.uint8)
            channels.append(frame_c)
        frames.append(cv2.merge(channels))
    
    return frames



In [10]:
def naive_blend(A, B, steps=60):
    frames = []
    for t in np.linspace(0, 1, steps):
        frame = (1-t)*A + t*B
        frame = np.clip(frame, 0, 255).astype(np.uint8)
        frames.append(frame)
    return frames

A, B = load_images("rigid2.png", "rigid1.png")

k = 1
print("Generating SVD frames...")
svd_forward = morph_color_k(A, B, steps=60, k=k)
svd_backward = svd_forward[::-1]
svd_frames = svd_forward + svd_backward

print("Generating naive frames...")
naive_forward = naive_blend(A, B, steps=60)
naive_backward = naive_forward[::-1]
naive_frames = naive_forward + naive_backward

print("Done! Press q to quit, +/- to change k")

while True:
    quit_flag = False
    recompute = False

    for svd_frame, naive_frame in zip(svd_frames, naive_frames):
        svd_display = svd_frame.copy()
        naive_display = naive_frame.copy()

        cv2.putText(svd_display, f'SVD Morph k={k}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(naive_display, 'Naive Blend', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        combined = np.hstack([naive_display, svd_display])
        cv2.imshow("Naive vs SVD Morph", combined)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            quit_flag = True
            break
        elif key == ord('+'):
            k = min(300, k + 50)
            recompute = True
            break
        elif key == ord('-'):
            k = max(1, k - 50)
            recompute = True
            break

    if quit_flag:
        break

    if recompute:
        print(f"Recomputing SVD with k={k}...")
        svd_forward =morph_color_k(A, B, steps=60, k=k)
        svd_backward = svd_forward[::-1]
        svd_frames = svd_forward + svd_backward

cv2.destroyAllWindows()

Generating SVD frames...
Generating naive frames...
Done! Press q to quit, +/- to change k
Recomputing SVD with k=51...
Recomputing SVD with k=101...
Recomputing SVD with k=151...
Recomputing SVD with k=201...
